# Human-in-the-loop — tool confirmation, end to end

Some tool calls should never fire without a human saying **yes** — deleting a
file, wiring money, sending an email. shipit lets a tool declare that for
itself with one decorator, and the permission gate **pauses** before it runs.

| § | What you'll see |
|---|---|
| 1 | Setup — this repo on the path |
| 2 | The `@requires_confirmation` decorator + the gate (no model) |
| 3 | A real agent that pauses for **your** approval (an HTML card) |
| 4 | How it works, end to end |

## 1 · Setup

Same guard as the other notebooks: this repo on the path, and an assertion if
some other installed `shipit_agent` wins.

In [ ]:
import sys
from pathlib import Path


def repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'shipit_agent' / '__init__.py').exists():
            return candidate
    raise RuntimeError(f'Could not locate the shipit_agent repo from {start}')


REPO = repo_root(Path.cwd())
sys.path.insert(0, str(REPO))
import shipit_agent
assert 'site-packages' not in shipit_agent.__file__, 'restart kernel — installed pkg loaded'
print('shipit-agent', shipit_agent.__version__)

## 2 · The decorator + the gate

`@requires_confirmation` marks a tool. `PermissionEngine.check()` then returns
`ASK` for it — **even under a bypass mode**, because the tool itself demands it.
Precedence: **deny > confirmation floor > mode (bypass/plan) > allow > ask**.

In [ ]:
from shipit_agent.tools import requires_confirmation
from shipit_agent.permissions import PermissionEngine
from shipit_agent.tools.base import ToolOutput


@requires_confirmation('Deletes a path permanently — cannot be undone.', impact='irreversible')
class DeleteTool:
    name = 'delete_path'
    description = 'Delete a file or directory.'

    def schema(self):
        return {'type': 'function', 'function': {'name': self.name,
                'description': self.description, 'parameters': {'type': 'object',
                'properties': {'path': {'type': 'string'}}, 'required': ['path']}}}

    def run(self, context=None, **kw):
        return ToolOutput(text=f"deleted {kw.get('path')}", metadata={})


# bypass would allow everything — but the tool floors it to ASK:
d = PermissionEngine(mode='bypass').check('delete_path', {'path': '/tmp/x.pdf'}, tool=DeleteTool)
print('decision :', d.decision.value)
print('reason   :', d.reason)

### Conditional — only the dangerous call pauses

A `when(arguments)` predicate lets a cheap call run free and stops only the
one that matters.

In [ ]:
@requires_confirmation('Large transfer needs sign-off.', impact='irreversible',
                       when=lambda a: a.get('amount', 0) >= 10_000)
class WireTransfer:
    name = 'wire_transfer'
    def schema(self):
        return {'type': 'function', 'function': {'name': self.name, 'parameters':
                {'type': 'object', 'properties': {'amount': {'type': 'number'},
                'to': {'type': 'string'}}, 'required': ['amount', 'to']}}}
    def run(self, context=None, **kw):
        return ToolOutput(text='sent', metadata={})


eng = PermissionEngine(mode='bypass')
print('$5      ->', eng.check('wire_transfer', {'amount': 5, 'to': 'x'}, tool=WireTransfer).decision.value)
print('$50,000 ->', eng.check('wire_transfer', {'amount': 50_000, 'to': 'x'}, tool=WireTransfer).decision.value)

## 3 · A real agent — you approve

Wire a `permission_callback` into the Agent. When the model calls the confirmed
tool, the gate renders an **HTML card** and **blocks** on `input()` until you
type `y` (allow — the tool runs) or `n` (deny — the agent is told it was
refused). That's the real human-in-the-loop.

The provider key path is read from a **local, gitignored `.env`**
(`VERTEX_SA_KEY=/path/to/service-account.json`) — never hardcoded here.

In [ ]:
import os, json as _json
from shipit_agent.llms.factory import load_env_file, build_llm_from_settings

load_env_file()   # reads a local, gitignored .env → sets VERTEX_SA_KEY etc.
VKEY = os.environ.get('VERTEX_SA_KEY') or os.environ.get('GOOGLE_APPLICATION_CREDENTIALS')
if not VKEY:
    raise SystemExit('Add to a local .env (gitignored):  VERTEX_SA_KEY=/path/to/service-account.json')
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = VKEY
os.environ['VERTEXAI_PROJECT'] = _json.load(open(VKEY))['project_id']
os.environ.setdefault('VERTEXAI_LOCATION', 'us-central1')
llm = build_llm_from_settings({'provider': 'vertex', 'model': 'vertex_ai/gemini-2.5-flash'}, load_env=False)
llm

In [ ]:
from shipit_agent import Agent
from shipit_agent.permissions import PermissionResult, PermissionDecision
from shipit_agent.tools.confirmation import confirmation_spec
from IPython.display import display, HTML

_TOOLS = {'delete_path': DeleteTool}
_COLOR = {'irreversible': '#ef4444', 'destructive': '#f59e0b', 'default': '#3b82f6'}


def human_approval(name, args):
    """Your human-in-the-loop: render a card, then block for a y/n decision."""
    spec = confirmation_spec(_TOOLS.get(name))
    impact = getattr(spec, 'impact', 'default') if spec else 'default'
    reason = spec.reason(name) if spec else ''
    c = _COLOR.get(impact, '#3b82f6')
    display(HTML(f'''
    <div style="border:1px solid {c};border-radius:12px;padding:16px;margin:8px 0;
                background:#0b0b0f;color:#e5e7eb;font-family:system-ui;max-width:560px">
      <div style="color:{c};font-weight:700;font-size:11px;text-transform:uppercase;letter-spacing:.12em">
        \U0001f514 confirmation required &middot; {impact}</div>
      <div style="margin-top:10px;font-size:15px">The agent wants to run
        <code style="color:{c};font-weight:700">{name}</code></div>
      <pre style="background:#000;border-radius:8px;padding:10px;margin-top:8px;font-size:12px">{args}</pre>
      <div style="margin-top:4px;color:#9ca3af;font-size:13px">{reason}</div></div>'''))
    if input(f'Approve {name}?  [y]es / [n]o : ').strip().lower() in ('y', 'yes'):
        return PermissionResult(PermissionDecision.ALLOW, reason='approved by human')
    return PermissionResult(PermissionDecision.DENY, reason='denied by human')


agent = Agent(llm=llm, tools=[DeleteTool()], permission_callback=human_approval,
              auto_use_skills=False, auto_project_memory=False, skill_source=None, max_iterations=4)

result = agent.run('Delete the file /tmp/old_report.pdf using the delete_path tool.')
print('\nANSWER:', result.output)

## 4 · How it works, end to end

1. The model emits a tool call.
2. Before running, the runtime calls `PermissionEngine.check(name, args, tool)`.
3. A **hard deny** always wins. Otherwise, if the tool was decorated with
   `@requires_confirmation` (and its `when` predicate applies), the gate
   consults your `permission_callback` — the human-in-the-loop.
4. `ALLOW` → the tool runs (with any edited arguments). `DENY` → it's skipped
   and the model is told. `ASK` with no callback → an approval **event** a UI
   draws a card from.
5. It is a **floor**: no bypass/allow mode can silently skip a tool that
   demands confirmation.